# Workspace Uptake & Retention — results

**This notebook holds no logic.** Every number comes from `analysis/uptake_lib.py`, which is importable and unit-tested outside Jupyter (`analysis/test_uptake_lib.py`), so any result here is reproducible without a kernel.

Pipeline:

```bash
./run.sh --experiment wur --job jobs/<id>          # runs the matrix
python3 lib/wur/aggregate.py --job-dir jobs/<id>   # -> jobs/<id>/analysis/*.parquet
python3 lib/wur/pilot_triage.py --job-dir jobs/<id> --markdown   # the §10 gates
jupyter lab analysis/uptake.ipynb
```

Read `analysis/PREREGISTRATION.md` first. Primary vs secondary, the exclusion policy and the decision rules were fixed before these data existed; nothing below is allowed to change them.

In [ ]:
import sys, os
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "analysis").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO / "analysis"))
sys.path.insert(0, str(REPO / "lib" / "wur"))

import pandas as pd
import uptake_lib as U

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
U.UPTAKE_LIB_VERSION

In [ ]:
# The pre-registered configuration. Changing any field after seeing data is a
# protocol deviation and must be declared as one in the report.
CFG = U.AnalysisConfig()
CFG.to_dict()

In [ ]:
# Point TABLES_DIR at an aggregate output dir. With none present the notebook
# runs end-to-end on seeded synthetic data so the plumbing is testable before the
# first real run exists — the banner below always says which one you are seeing.
TABLES_DIR = os.environ.get("WUR_TABLES_DIR", "")
if TABLES_DIR and (Path(TABLES_DIR) / "fact_trace.parquet").exists():
    T = U.load_tables(TABLES_DIR)
    SOURCE = f"REAL DATA: {TABLES_DIR}"
else:
    T = U.synthetic_tables(n_tasks=12, n_reps=5,
                           arms=("ctrl", "d1", "d2", "d3", "d2-check", "d2-table"),
                           read_rate_by_arm={"ctrl": 0.0, "d1": 0.85, "d2": 0.65, "d3": 0.35,
                                             "d2-check": 0.70, "d2-table": 0.68},
                           use_rate_ctrl=0.05,
                           use_lift_by_arm={"d1": 0.30, "d2": 0.22, "d3": 0.10,
                                            "d2-check": 0.26, "d2-table": 0.24},
                           gamma=0.4)
    SOURCE = "SYNTHETIC PLACEHOLDER — not results"
print(SOURCE); T

In [ ]:
# Exclusions, printed before any estimate. A silent exclusion is how a funnel
# study lies to itself.
DF, EXCL = U.analysis_frame(T, CFG)
EXCL.to_frame()

## 1. The funnel

`available -> read -> used -> retained`. Read-but-not-used is decorative; used-but-not-retained needs re-reading every turn; never-read is invisible regardless of quality. Three different failures, three different fixes.

In [ ]:
U.funnel_table(DF, CFG)

In [ ]:
U.plot_funnel(DF, CFG);

## 2. Read rate

PRIMARY is D4: inbound channels ∪ `self_thinking`. `read_inbound_only` is the mandatory sensitivity row and appears in every table where read is a denominator. Unknown reads (a truncation or the 256 KB ceiling hit a call targeting the fact file) leave the denominator and are bracketed, never coerced to 0.

In [ ]:
U.read_rate_table(DF, CFG)

In [ ]:
U.plot_read_rate(DF, CFG);

In [ ]:
# read - opened: did the agent FIND the fact, or stumble into it via grep?
U.incidental_exposure(DF, CFG)

## 3. Use — always as lift over the paired control

λ_{a,t} = mean_k used[a,t,k] − mean_k used[ctrl,t,k]; λ_a = mean_t λ_{a,t}, tested by a paired t across tasks. **n is the number of tasks, not the number of runs.** Raw rates are the appendix.

In [ ]:
LIFT = U.use_lift(DF, CFG)
LIFT

In [ ]:
# The conditional companion: `eligible` runs only. §4.3 requires both, together.
U.use_lift(DF, CFG, conditional=True)

In [ ]:
U.plot_lift(LIFT);

In [ ]:
# APPENDIX ONLY: raw use rates, unconditional and conditional on eligibility.
U.use_rate_table(DF, CFG)

## 4. Retention — sustained self-report under repeated elicitation

Not memory decay: the probe re-injects the nonce roughly every two tool calls, and `n_reinjections` is the dose covariate. Time is probe index. Summary statistic is **RMST over a common horizon J**, never the KM median, which is undefined whenever Ŝ(j) > 0.5 — the expected case. The reference arm is *not* the control: `ctrl` is fact-free and contributes zero rows by construction.

In [ ]:
RET = U.retention_table(DF, CFG)
print(f"J = {RET.horizon_J}   exploratory = {RET.exploratory}")
print(RET.note)
RET.per_arm

In [ ]:
U.plot_km(RET.dataset, horizon=RET.horizon_J);

In [ ]:
# Reported separately: after the agent has discharged a fact, dropping it is
# correct behaviour, not forgetting.
U.post_discharge_persistence(DF, CFG)

## 5. Mention, and the instrument's own health

Mention rate is P(named in ≥1 probe | exposed). Probe fidelity is the only metric here that generalizes past workspace design: if `affects_next_action` predicts the next tool call, cheap self-report is a legitimate monitoring instrument. `parse_ok ≥ 0.90` and `refused == 0` are pilot gates. With one fact per task, ≥2 slots are filler by construction — **the filler distribution is itself a finding**.

In [ ]:
U.mention_rate(DF, CFG)

In [ ]:
display(U.probe_quality(T, CFG))
display(U.probe_fidelity(T, CFG))
U.slot_class_distribution(T)

## 6. The alarms

Not findings. `confab_rate > 0.05` invalidates every exposure-conditioned metric; `unexplained_possession > 0.05` is a fixture-wide failure; `|φ(used, success)| > 0.8` means the battery is testing the mandate and the funnel has collapsed to one measurement. All three are computed on a frame that KEEPS quarantined rows — a quarantined control row that names the nonce *is* the confabulation.

In [ ]:
print(U.confabulation_rate(T, CFG))
print(U.unexplained_possession_rate(T, CFG))
U.orthogonality_phi(DF, CFG)

## 7. Depth, format, and how much the probe changed the answer

Depth prices "how deep can documentation go"; format asks whether presentation beats placement. Both are cluster-level first (one slope or one contrast per task, t across tasks); the mixed-effects logistic is secondary and its asymptotics at 12 clusters are a promise, not a fact. Probe reactivity bounds every absolute number in this notebook.

In [ ]:
DEPTH = U.depth_sensitivity(DF, CFG)
print(DEPTH["slope"])
print(DEPTH["secondary_model"])
display(DEPTH["contrasts"])
U.plot_depth(DEPTH);

In [ ]:
FMT = U.format_sensitivity(DF, CFG)
print(FMT["omnibus"])
display(FMT["contrasts"])
U.probe_reactivity(DF, CFG)

## 8. Secondary tests, and the rule that suppresses them

CMH and within-task label permutation condition on the stratum margins, so they test the sharp null. Under task × arm heterogeneity they are anti-conservative — measured type-I 0.029 → 0.135 (CMH) and 0.030 → 0.139 (permutation) as γ goes 0 → 2 at p₀ = 0.5, while the primary paired t stays at 0.047 → 0.053. §12 therefore suppresses both above γ̂ = 0.5, and `uptake_lib` refuses to emit their p-values there.

In [ ]:
ARM, CTRL, OUTCOME = "d2", CFG.baseline_condition, "used"
print(U.gamma_hat(DF, ARM, CTRL, OUTCOME))
print(U.cmh_test(DF, ARM, CTRL, OUTCOME, CFG))
print(U.permutation_test_within_task(DF, ARM, CTRL, OUTCOME, CFG, draws=2000))

## 9. What this design could detect

`analysis/power_results.json` is the frozen output of `python3 analysis/power.py`. Re-run it against the pilot's own parameters with `--pilot-dir jobs/<id>/analysis`.

In [ ]:
import json
POWER = json.loads((REPO / "analysis" / "power_results.json").read_text())
display(pd.DataFrame(POWER["mdd"]))
pd.DataFrame(POWER["type_i"])

In [ ]:
# One call for everything above — what analysis/REPORT.md is written from.
S = U.summarize(T, CFG)
print(SOURCE)
print({k: (type(v).__name__) for k, v in S.items()})
S["funnel"]